In [3]:
import torch
import numpy as np
import pickle
import rasterio
import psutil
import gc
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from sklearn.preprocessing import StandardScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# load checkpoint
ckpt = torch.load('../data/canwell_sage_nobasin.pt', weights_only=False)
y_mean    = ckpt['y_mean']
y_std     = ckpt['y_std']
shape     = ckpt['shape']
transform = ckpt['transform']
node_to_idx = ckpt['node_to_idx']
best_model  = ckpt['best_model']

# load features
feat     = torch.load('../data/canwell_features.pt')
x        = feat['x']
y        = feat['y']
is_slope = feat['is_slope']
is_basin = feat['is_basin']
has_diff = feat['has_diff']
del feat
gc.collect()

print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

RAM: 16.4%


In [6]:
# normalize features
x_np = x.numpy()
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_np)

# rebuild masks
slope_with_diff = (is_slope & has_diff).nonzero(as_tuple=True)[0]
perm = torch.randperm(len(slope_with_diff), generator=torch.Generator().manual_seed(42))
n = len(slope_with_diff)
train_idx = slope_with_diff[perm[:int(0.70*n)]]
val_idx   = slope_with_diff[perm[int(0.70*n):int(0.85*n)]]
test_idx  = slope_with_diff[perm[int(0.85*n):]]

is_masked = torch.zeros(x.shape[0], dtype=torch.float)
is_masked[train_idx] = 1.0
is_masked[val_idx]   = 1.0
is_masked[test_idx]  = 1.0

# no basin signal
diff_context = torch.zeros(x.shape[0], dtype=torch.float)

x_final = torch.cat([
    torch.tensor(x_scaled, dtype=torch.float),
    is_masked.unsqueeze(1),
    diff_context.unsqueeze(1)
], dim=1)

# load edge index and build data
edge_index = torch.load('../data/canwell_edgidx.pt')
y_norm = (y - y_mean) / y_std

data = Data(x=x_final, edge_index=edge_index, y=y_norm)
data.is_slope = is_slope
data.has_diff = has_diff

# free what we don't need
del edge_index, x_scaled, x_np, diff_context, is_masked
gc.collect()

print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

RAM: 21.2%


In [7]:
class CanwellSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=64):
        super(CanwellSAGE, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.conv3 = SAGEConv(hidden_channels, hidden_channels)
        self.lin1 = Linear(hidden_channels, hidden_channels // 2)
        self.lin2 = Linear(hidden_channels // 2, 1)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.relu(self.conv3(x, edge_index))
        x = F.relu(self.lin1(x))
        x = self.lin2(x)
        return x.squeeze(1)
        

In [8]:
model = CanwellSAGE(in_channels=6, hidden_channels=64).to(device)
model.load_state_dict(best_model)
model.eval()

all_slope_idx = (is_slope & has_diff).nonzero(as_tuple=True)[0]

all_slope_loader = NeighborLoader(
    data, num_neighbors=[10,10,10],
    batch_size=512,
    input_nodes=all_slope_idx,
    shuffle=False,
)

all_preds = []
with torch.no_grad():
    for batch in all_slope_loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)
        all_preds.append(out[:batch.batch_size].cpu())

all_preds_m = torch.cat(all_preds).numpy() * y_std + y_mean

print(f"Predictions: {len(all_preds_m):,}")
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

/home/samuelnwalters/miniconda3/envs/gd_env/lib/python3.10/site-packages/torch_geometric/loader/neighbor_loader.py:229: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(


Predictions: 2,838,168
RAM: 26.9%


In [9]:
# free data and loader before loading graph
del data, all_slope_loader
gc.collect()
print(f"RAM after cleanup: {psutil.virtual_memory().percent:.1f}%")

RAM after cleanup: 19.2%


In [10]:
# rebuild idx_to_demidx
with open('../data/canwell_graph.pkl', 'rb') as f:
    graph_data = pickle.load(f)

idx_to_demidx = {}
for node_id, attr in graph_data['G'].nodes(data=True):
    idx = node_to_idx[node_id]
    idx_to_demidx[idx] = attr['dem_idx']

del graph_data['G']
gc.collect()
print(f"RAM after graph load: {psutil.virtual_memory().percent:.1f}%")

RAM after graph load: 41.0%


In [11]:
# rasterize
recon_nobasin = np.full(shape, np.nan, dtype=np.float32)
for i, node_id in enumerate(all_slope_idx.numpy()):
    r, c = idx_to_demidx[node_id]
    recon_nobasin[r, c] = all_preds_m[i]

print(f"Filled: {np.sum(~np.isnan(recon_nobasin)):,} pixels")

# save
with rasterio.open('canwell_gnnnobasin_nobasin_northslopediff.tif', 'w',
    driver='GTiff', dtype='float32', width=shape[1], height=shape[0],
    count=1, crs=graph_data['crs'], transform=transform, nodata=float('nan')) as dst:
    dst.write(recon_nobasin, 1)

print("Saved!")
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

Filled: 2,838,168 pixels
Saved!
RAM: 41.3%
